In [4]:
import pandas as pd


def read_protocol(path):
    """Read protocol file where col0 (filepath) may contain spaces."""
    with open(path) as f:
        data = [line.strip().rsplit(' ', 2) for line in f if line.strip()]
    return pd.DataFrame(data)


merge_protocol = read_protocol('/nvme2/hungdx/Lightning-hydra/logs/results/Kipot_benchmark/MDT_241214_lora_250501/merged_protocol_cnsl_lora_elevenlabs_xlsr_conformertcm_mdt_more_elevenlabs_MDT_241214_lora_250501.txt')

# MDT_LoRA (May 2025)

mdt_lora_may2025 = read_protocol('/nvme2/hungdx/Lightning-hydra/logs/results/Kipot_benchmark/MDT_241214_lora_250501/merged_scores_cnsl_lora_elevenlabs_xlsr_conformertcm_mdt_more_elevenlabs_MDT_241214_lora_250501.txt')

# MDT (Feb 2026) VAD
mdt_vad_feb2026 = read_protocol('/nvme2/hungdx/Lightning-hydra/logs/results/Kipot_benchmark/06feb26_xlsr_conformertcm_mdt/merged_scores_cnsl_lora_elevenlabs_xlsr_conformertcm_mdt_lora_infer_06feb26_xlsr_conformertcm_mdt.txt')

# MDT_LoRA (Feb 2026) VAD + w/o VAD
mdt_lora_vad_w_o_vad_feb2026 = read_protocol('/nvme2/hungdx/Lightning-hydra/logs/results/Kipot_benchmark/26feb26_xlsr_conformertcm_mdt_lora/merged_scores_cnsl_lora_elevenlabs_xlsr_conformertcm_mdt_more_elevenlabs_26feb26_xlsr_conformertcm_mdt_lora.txt')


In [6]:
# Set column names
merge_protocol.columns = ['filepath', 'subset', 'label']
mdt_lora_may2025.columns = ['filepath', 'spoof_score', 'bonafide_score']
mdt_vad_feb2026.columns = ['filepath', 'spoof_score', 'bonafide_score']
mdt_lora_vad_w_o_vad_feb2026.columns = ['filepath', 'spoof_score', 'bonafide_score']

In [8]:
# merge each model with protocol
mdt_lora_may2025 = pd.merge(mdt_lora_may2025, merge_protocol, on='filepath', how='left')
mdt_vad_feb2026 = pd.merge(mdt_vad_feb2026, merge_protocol, on='filepath', how='left')
mdt_lora_vad_w_o_vad_feb2026 = pd.merge(mdt_lora_vad_w_o_vad_feb2026, merge_protocol, on='filepath', how='left')


mdt_lora_may2025

,filepath,spoof_score,bonafide_score,subset,label
0,2025_Kipot/2025/eval/a00/1001/C0002-1001F1221-...,-4.642345428466797,4.632585048675537,eval,bonafide
1,2025_Kipot/2025/eval/a00/1001/C0003-1001F1221-...,-2.1109485626220703,1.884427547454834,eval,bonafide
2,2025_Kipot/2025/eval/a00/1001/C0004-1001F1221-...,-0.6217828392982483,-0.4989219605922699,eval,bonafide
3,2025_Kipot/2025/eval/a00/1001/C0006-1001F1221-...,-0.959135115146637,0.07091904431581497,eval,bonafide
4,2025_Kipot/2025/eval/a00/1001/C0008-1001F1221-...,-2.05877423286438,1.5347117185592651,eval,bonafide
...,...,...,...,...,...
4442182,wav_segments/202505131853541193_00000_seg015.wav,-3.159695625305176,2.6706089973449707,eval,bonafide
4442183,wav_segments/202505091057177839_00000_seg040.wav,-2.8696956634521484,2.4078803062438965,eval,bonafide
4442184,wav_segments/202505011723258143_00000_seg024.wav,-2.390042543411255,2.0492734909057617,eval,bonafide
4442185,wav_segments/202505081258393891_00000_seg034.wav,-4.337125301361084,4.362480640411377,eval,bonafide


In [10]:
filter_keywords = ['veo3', 'sora', 'seedance']

models = {
    'MDT_LoRA (May 2025)': mdt_lora_may2025,
    'MDT (Feb 2026) VAD': mdt_vad_feb2026,
    'MDT_LoRA (Feb 2026)': mdt_lora_vad_w_o_vad_feb2026,
}

def calc_accuracy(df):
    df = df.copy()
    df['spoof_score'] = df['spoof_score'].astype(float)
    df['bonafide_score'] = df['bonafide_score'].astype(float)
    df['pred'] = (df['bonafide_score'] > df['spoof_score']).map({True: 'bonafide', False: 'spoof'})
    return (df['pred'] == df['label']).mean() * 100

results = []
for model_name, df in models.items():
    for keyword in filter_keywords:
        subset = df[df['filepath'].str.contains(keyword, case=False)]
        if len(subset) == 0:
            continue
        acc = calc_accuracy(subset)
        results.append({
            'Model': model_name,
            'Subset': keyword,
            'Accuracy (%)': round(acc, 2),
            'Num Samples': len(subset),
        })

results_df = pd.DataFrame(results)
results_df.pivot(index='Subset', columns='Model', values='Accuracy (%)')


Model,MDT (Feb 2026) VAD,MDT_LoRA (Feb 2026),MDT_LoRA (May 2025)
Subset,,,
seedance,95.00,95.00,45.00
sora,99.84,99.84,66.56
veo3,97.42,98.71,44.27


In [11]:
results_df

,Model,Subset,Accuracy (%),Num Samples
0,MDT_LoRA (May 2025),veo3,44.27,698
1,MDT_LoRA (May 2025),sora,66.56,610
2,MDT_LoRA (May 2025),seedance,45.00,20
3,MDT (Feb 2026) VAD,veo3,97.42,698
4,MDT (Feb 2026) VAD,sora,99.84,610
5,MDT (Feb 2026) VAD,seedance,95.00,20
6,MDT_LoRA (Feb 2026),veo3,98.71,698
7,MDT_LoRA (Feb 2026),sora,99.84,610
8,MDT_LoRA (Feb 2026),seedance,95.00,20
